In [ ]:
from rocketcea.cea_obj_w_units import CEA_Obj
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field, fields

def q(unit, desc="", fmt=".4g", **kw):
    """Declare a physical quantity field with its unit."""
    return field(metadata={"unit": unit, "desc": desc, "fmt": fmt}, **kw)


class PrettyPrint:
    def __str__(self):
        rows = []
        for f in fields(self):
            v = getattr(self, f.name)
            if v is None:
                continue
            s = format(v, f.metadata.get("fmt", ".4g")) if isinstance(v, (int, float)) else str(v)
            rows.append((f.name, s, f.metadata.get("unit", ""), f.metadata.get("desc", ""), f.init))

        wn = max(len(r[0]) for r in rows)
        wv = max(len(r[1]) for r in rows)
        out = [type(self).__name__]
        for is_input, label in ((True, "inputs"), (False, "derived")):
            sel = [r for r in rows if r[4] is is_input]
            if not sel:
                continue
            out.append(f"  [{label}]")
            out += [f"    {n:<{wn}} = {s:>{wv}} {u:<6} {d}" for n, s, u, d, _ in sel]
        return "\n".join(out)


@dataclass
class RocketEngine(PrettyPrint):
    F:  float = q("N",    "thrust")
    OF: float = q("-",    "mixture ratio")
    p_c_bar: float = q("bar", "chamber pressure")
    p_e_bar: float = q("bar", "exit pressure")
    p_a_bar: float = q("bar", "ambient pressure")
    fuel_name: str = "RP-1"
    oxidizer_name: str = "LOX"
    eff_cstar: float = q("-", "c* efficiency", default=0.95)

    # outputs — declared, so the printer finds them
    eps:      float = q("-",    "area ratio",     init=False, default=None)
    isp_sl:   float = q("s",    "sea-level Isp",  init=False, default=None)
    mdot:     float = q("kg/s", "total mdot",     init=False, default=None)
    mdot_ox:  float = q("kg/s", "oxidiser mdot",  init=False, default=None)

    def _calculate_from_thrust(self):
        self.cea = CEA_Obj(
            oxName = self.oxidizer_name,
            fuelName = self.fuel_name,
            isp_units='sec',
            cstar_units = 'm/s',
            pressure_units='Bar',
            temperature_units='K',
            sonic_velocity_units='m/s',
            enthalpy_units='J/g',
            density_units='kg/m^3',
            specific_heat_units='J/kg-K',
            viscosity_units='centipoise', # stored value in pa-s
            thermal_cond_units='W/cm-degC', # stored value in W/m-K
            # fac_CR=self.cr,
            make_debug_prints=False)

        # With low pressure engines, I will hereon assume frozen at throat properties. Refer to notes on non equilibrium flows if this assumption is not valid.
        self.eps = self.cea.get_eps_at_PcOvPe(Pc=self.p_c_bar, MR=self.OF, PcOvPe=self.p_c_bar/self.p_e_bar, frozen=1, frozenAtThroat=1)
        # self.Cf_design = self.cea.getFrozen_PambCf(Pamb=self.p_a_bar, Pc=self.p_c_bar, eps=self.eps, frozenAtThroat=1)
        self.isp_sl = self.cea.estimate_Ambient_Isp(Pc=15,MR=self.OF,eps=self.eps,Pamb=1.01325)[0] * self.eff_cstar
        # self.t_c = self.cea.get_Tcomb(Pc=15, MR=OF)
        self.mdot = self.F / (self.isp_sl * 9.81)
        self.mdot_ox = self.mdot / (1 + self.OF)
        self.mdot_fuel = self.mdot - self.mdot_ox
    # Static method to make an engine object from thrust and OF
    @staticmethod
    def generate_from_thrust(F: float, OF: float, p_c_bar: float, p_e_bar: float, fuel_name: str = 'RP-1', oxidizer_name: str = 'LOX', eff_cstar: float = 0.95):
        engine = RocketEngine(F, OF, p_c_bar, p_e_bar, 1.01325, fuel_name, oxidizer_name, eff_cstar)
        engine._calculate_from_thrust()
        return engine

        
engine = RocketEngine.generate_from_thrust(F=15000, OF=2.5, p_c_bar=50, p_e_bar=1, fuel_name='RP-1', oxidizer_name='LOX', eff_cstar=0.95)
print(engine)

@dataclass
class Pump(PrettyPrint):
    N: float = q("rpm", "rotational speed")
    Q: float = q("m^3/s", "volumetric flow rate")


USER_HOME_DIR=C:\Users\Martin
RocketEngine
  [inputs]
    F             = 1.5e+04 N      thrust
    OF            =     2.5 -      mixture ratio
    p_c_bar       =      50 bar    chamber pressure
    p_e_bar       =       1 bar    exit pressure
    p_a_bar       =   1.013 bar    ambient pressure
    fuel_name     =    RP-1        
    oxidizer_name =     LOX        
    eff_cstar     =    0.95 -      c* efficiency
  [derived]
    eps           =   6.714 -      area ratio
    isp_sl        =   218.9 s      sea-level Isp
    mdot          =   6.985 kg/s   total mdot
    mdot_ox       =   1.996 kg/s   oxidiser mdot
